# 🚀 Google Colab Pro Scheduled Runner (Dual-Stream Ingestion & Training)
### Kết Hợp Option 1 (Scheduled Headless Daemon) & Form Điều Khiển Option B

Notebook này được thiết kế theo chuẩn **Google Colab Pro Scheduled Runs** (chạy tự động ngầm) đồng thời tích hợp **Interactive Form Controls (#@param)** cho phép nghiên cứu viên thử nghiệm thủ công bất kỳ lúc nào.

### 📌 Phân Tách 2 Mô Hình Độc Lập:
1. **Mô Hình 1 (General Vision & Signal Model)**: Giữ nguyên tập dataset tổng quát (`NTIRE-RobustAIGenDetection`, `CIFAKE`, `DF40`, `DFFD`) + Khai phá mẫu biên khó (Hard Negatives $H(P) \ge 0.70$) từ Meta Threads.
2. **Mô Hình 2 (Human Biometric Multi-Modal Model)**: Bổ sung **4 Dataset Human Chuyên Biệt** (`MultiFakeVerse`, `HiDF`, `140k Real & Fake Faces`, `DeepDetect-2025/2026`) + Quota In-The-Wild Meta Threads (10,000 ảnh, 500 video, 500 audio qua 5 phân lớp sinh trắc người Việt).
3. **Báo Cáo Huấn Luyện Toàn Diện**: Tự động xuất file `training_report_dual_stream.md` lưu trữ trực tiếp vào Google Drive (`/content/drive/MyDrive/Thread_Data/reports/`).

--- 
## Bước 1: Kết Nối Google Drive (Auto-Mount Without Pop-up)

In [ ]:
import os, sys, time
from google.colab import drive

# Auto-mounts Google Drive at /content/drive/MyDrive without pop-up blocking
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive đã được gắn kết thành công tại /content/drive/MyDrive')

--- 
## Bước 2: Cập Nhật Mã Nguồn Từ GitHub & Cài Đặt Thư Viện

In [ ]:
# Pull mã nguồn mới nhất từ GitHub
!git clone https://github.com/ThangCao106/DeepFake-Detection-Agent.git /content/DeepFake-Detection-Agent || (cd /content/DeepFake-Detection-Agent && git fetch origin main && git reset --hard origin/main && git pull origin main)
%cd /content/DeepFake-Detection-Agent

# Cài đặt các thư viện cần thiết ở chế độ im lặng
!pip install -q -r requirements.txt transformers timm
print('✅ Môi trường và kho mã nguồn đã sẵn sàng.')

--- 
## Bước 3: Form Cấu Hình Tương Tác (Option B Form UI) & Lựa Chọn Tham Số

In [ ]:
#@title 🎛️ Bảng Điều Khiển Huấn Luyện & Thu Thập Dữ Liệu { run: "auto" }

#@markdown ### 🔄 Chế Độ Thực Thi (Execution Mode)
RUN_MODE = "SCHEDULED_BATCH_RUN" #@param ["SCHEDULED_BATCH_RUN", "MANUAL_TEST_RUN", "CONTINUOUS_DAEMON_4H"]
LOOP_INTERVAL = "4h" #@param ["1h", "2h", "4h", "8h", "12h", "24h"]

#@markdown ### 🎯 Lựa Chọn Mô Hình Huấn Luyện (Model Selection)
TRAIN_BOTH_MODELS = True #@param {type:"boolean"}
TRAIN_MODEL_1_GENERAL = True #@param {type:"boolean"}
TRAIN_MODEL_2_HUMAN = True #@param {type:"boolean"}

#@markdown ### 📊 Quota Dữ Liệu In-The-Wild Meta Threads (Mỗi Chu Kỳ)
CRAWL_IMAGES = 10000 #@param {type:"slider", min:100, max:20000, step:100}
CRAWL_VIDEOS = 500 #@param {type:"slider", min:50, max:2000, step:50}
CRAWL_AUDIOS = 500 #@param {type:"slider", min:50, max:2000, step:50}
INSPECTION_SAMPLES = 25 #@param {type:"integer"}

#@markdown ### 🏋️ Siêu Tham Số Huấn Luyện (Hyperparameters)
EPOCHS = 3 #@param {type:"integer"}
BATCH_SIZE = 32 #@param {type:"integer"}
LEARNING_RATE = 0.0001 #@param {type:"number"}
ANTI_IDLE_HEARTBEAT = True #@param {type:"boolean"}

print(f"[Cấu hình] Chế độ: {RUN_MODE} | Crawl: {CRAWL_IMAGES} ảnh, {CRAWL_VIDEOS} video, {CRAWL_AUDIOS} audio")
print(f"[Cấu hình] Huấn luyện: Model 1={TRAIN_MODEL_1_GENERAL}, Model 2={TRAIN_MODEL_2_HUMAN} (Both={TRAIN_BOTH_MODELS})")
print(f"[Cấu hình] Tham số: Epochs={EPOCHS}, BatchSize={BATCH_SIZE}, LR={LEARNING_RATE}")

--- 
## Bước 4: Thực Thi Đồng Bộ Dữ Liệu & Huấn Luyện 2 Mô Hình

In [ ]:
# Xây dựng câu lệnh thực thi dựa trên Form phía trên
cmd_args = [
    f"--images {CRAWL_IMAGES}",
    f"--videos {CRAWL_VIDEOS}",
    f"--audios {CRAWL_AUDIOS}",
    f"--samples {INSPECTION_SAMPLES}",
    f"--epochs {EPOCHS}",
    f"--batch_size {BATCH_SIZE}",
    f"--lr {LEARNING_RATE}"
]

if TRAIN_BOTH_MODELS:
    cmd_args.append("--train-both-models")
else:
    if TRAIN_MODEL_1_GENERAL:
        cmd_args.append("--train-model1")
    if TRAIN_MODEL_2_HUMAN:
        cmd_args.append("--train-model2")

if RUN_MODE == "CONTINUOUS_DAEMON_4H":
    cmd_args.append(f"--loop {LOOP_INTERVAL}")
    if ANTI_IDLE_HEARTBEAT:
        cmd_args.append("--keep-alive")

cmd_str = "python colab_sync_to_gdrive.py " + " ".join(cmd_args)
print(f"🚀 Đang thực thi: {cmd_str}\n")
!{cmd_str}

--- 
## Bước 5: Báo Cáo Huấn Luyện Toàn Diện (Report Data Đầy Đủ)

In [ ]:
from pathlib import Path
import IPython.display as display

report_path = Path("report_colab/training_report_dual_stream.md")
gdrive_report = Path("/content/drive/MyDrive/Thread_Data/reports/training_report_dual_stream.md")

target_report = gdrive_report if gdrive_report.exists() else report_path

if target_report.exists():
    print(f"📋 Hiển thị Báo Cáo Huấn Luyện Toàn Diện từ: {target_report}\n")
    with open(target_report, "r", encoding="utf-8") as f:
        report_content = f.read()
    display.display(display.Markdown(report_content))
else:
    print("⚠️ Chưa tìm thấy file báo cáo training_report_dual_stream.md. Vui lòng kiểm tra lại log thực thi.")

# Kiểm tra các tệp checkpoint trọng số đã lưu trên Google Drive
print("\n" + "="*65)
print("💾 DANH MỤC TRỌNG SỐ ĐÃ LƯU TRÊN GOOGLE DRIVE /content/drive/MyDrive/Thread_Data/checkpoints/")
print("="*65)
ckpt_dir = Path("/content/drive/MyDrive/Thread_Data/checkpoints")
if ckpt_dir.exists():
    for p in ckpt_dir.iterdir():
        size_mb = round(p.stat().st_size / (1024 * 1024), 2)
        print(f"  - [{p.name}] Kích thước: {size_mb} MB")
else:
    print("  (Đang chạy local - kiểm tra thư mục report_colab/gdrive_storage/checkpoints/)")
print("="*65)